# Lesson 2: Image Processing — Blurring, Thresholding, and Edges

This what makes face detection possible. Detectors don't look at a clean nice photo, they usually work on simplified processed versions of the frames. So we're building the toolkit that comes before detection.

# Blurring (Noise Reduction):

Real camera footage is noisy, the pixel-level flickers which confuses the detector. Blurring smoothes it out by averaging pixels with their neighbour.

In [1]:
import cv2

frame = cv2.imread('Screenshot (240).png')    # loads as Blue, Green, Red (BGR) array
blurred = cv2.GaussianBlur(frame, (5, 5), 0)   # Gaussian blur with 5x5 kernel

cv2.imshow('Original', frame)
cv2.imshow('Blurred', blurred)
cv2.waitKey(0)
cv2.destroyAllWindows()


- **Kernel:** A small grid of numbers that slides over every pixel on the image. For blurring, it's a grid that says "take the average of everything in this window."

- (5, 5) is the kernel size — a 5x5 window of pixels gets averaged together for every pixel in the image. Bigger kernel = blurrier, but also slower and can wash out real detail (like edges of a face). 

- Bigger kernel = blurrier

---

## 2. Thresholding (turning gray into black-and-white)

Thresholding takes a grayscale image and forces every pixel to be either pure black or pure whhite based on a cutoff value. This vital for **isolating shapes from the backgrounds.**

In [4]:
gray = cv2.imread('output.jpg')
_, thresh = cv2.threshold(gray, 127, 255,cv2.THRESH_BINARY)
cv2.imshow('Thresholded', thresh)
cv2.waitKey(0)
cv2.destroyAllWindows()


**_, thresh = cv2.threshold(gray, 127, 255, cv2.THRESH_BINARY):**

- Any pixel >=127 -> 255, anything darker (<=) becomes 0
- The **_** throws away the first return value(it echoes back the theshold used)

### Adaptive Thresholding:

Recalculates the cutoff for different regions of teh image, better when lighting isn't even accross the frame (webcam)

In [ ]:
adaptive =cv2.adaptiveThreshold(gray, 255, cv2, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 11, 2)

- Parameters:
    (gray_image, 
    value_for_px_over_threshold, 
    cv2.ADAPTIVE_THRESH_GAUSSIAN_C: method used to calculate, 
    px_greater_become_255_lower_become_0, 
    px_matrix_size,
    this is C, a constant that gets subtracted from the calculated local average before it's used as the cutoff. It's a fine-tuning knob:)

**Putting it together in plain English**
For every single pixel: "Look at the 11x11 box of pixels around me, take their Gaussian-weighted average, subtract 2 from that average — that's my personal cutoff. If I'm brighter than my own cutoff, I become white. Otherwise, black."

## 3. Edge detection (Canny)

This finds the outline of objects, it looks for anywhere the brightness changes sharply.


In [10]:
edges = cv2.Canny(gray, 50, 100)
cv2.imshow('Edges', edges)
cv2.waitKey(0)
cv2.destroyAllWindows()

- The 2 numbers are lower/upper thresholds "how strong the brightness change counts as an edge"
- Lower Values -> More edges detected (including noise)
- Higher Values -> Less more obvious edges detected
---

## Contours (turning edges into shapes you can use)

Edges are just pixels. Contours group your edge pixels into shapes that code can measure, count, draw boxes around (Object Detection)


In [9]:
contours, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
cv2.drawContours(frame, contours, -1, (0, 255, 0), 2)
cv2.imshow('Contours', frame)
cv2.waitKey(0)
cv2.destroyAllWindows()

**contours:** This a list of arrays, each one is a set of points outlining one detected shape. -1 means "draw on all of them."

**cv2.findContours:**

- Scan the edges image (from canny) and groups connected white pixels into white distinct outlines.
- Each edge becomes a contour (an x,y list tracing the boundary of the shape)

- **cv2.RETR_EXTERNAL:** Controls which contours get returned (Outmost contours)
- **cv2.CHAIN_APPROX_SIMPLE:** Control the amount of points needed the define a shape.

- **Return Values:** contours, _ = cv2.findContours(...) -> Returns 2 points
    - contours — a Python list, where each item is a NumPy array of (x, y) points outlining one detected shape. If your frame has 4 distinct blobs, contours will have 4 arrays in it. len(contours) tells you how many separate shapes were found.

    - _ — the second return value is the hierarchy info (parent/child relationships between contours, relevant if you'd used RETR_TREE). We're throwing it away with _ since RETR_EXTERNAL makes it not useful here.

**cv2.drawContours():**

- Paints the contour coordinates onto a real image

- **frame** — the image to draw on
- **contours** — the full list of contour arrays we just found.
- **-1** — this is the contour index to draw. Since contours is a list, you could tell it to draw only contours[0] (just the first shape) by passing 0. Passing -1 means "draw all of them."
- **2** — line thickness in pixels.
---

## MINI Project 2

In [4]:
import cv2

cap = cv2.VideoCapture(0, cv2.CAP_DSHOW)  # Use CAP_DSHOW to avoid warning on Windows
live = True

while live:
    ret, frame = cap.read()
    if not ret:
        break
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    blurred = cv2.GaussianBlur(gray, (15, 15), 0)
    edges = cv2.Canny(blurred, 50, 150)
    contours, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    cv2.drawContours(frame, contours, -1, (0, 255, 0), 2)

    cv2.imshow('Live Frame', frame)
    

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break
cap.release()
cv2.destroyAllWindows()